<a href="https://colab.research.google.com/github/efecclick/qwen3/blob/main2/offroad_detection_karsilastirma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Off-road object detection - VLM karşılaştırması

Off-road (bounding box'lı) bir veri seti üzerinde üç **vision-language modelini** (VLM) kıyaslıyorum.
Hepsi prompt ile grounding yapar: sınıf isimlerini verirsin, kutu koordinatlarını üretirler.

Modeller:
- **Qwen2.5-VL-3B-Instruct**  (Qwen/Qwen2.5-VL-3B-Instruct)
- **InternVL2.5-4B**  (OpenGVLab/InternVL2_5-4B)
- **Llama-3.2-11B-Vision-Instruct**  (meta-llama/Llama-3.2-11B-Vision-Instruct)

Ölçtüklerim: **latency, FPS, GPU bellek, model boyutu, mAP@0.5, precision/recall/IoU**.

> Her VLM'in koordinat formatı farklı (Qwen mutlak piksel, InternVL [0,1000] normalize, Llama belirsiz),
> o yüzden her birine ayrı parser + debug hücresi var. İlk çalıştırmada debug çıktılarına bak.

In [16]:
%pip install -q -U "transformers>=4.57.0" accelerate bitsandbytes qwen-vl-utils timm einops pyyaml pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.7 MB/s eta 0:00:00


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, re, json, glob, time
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display

from transformers import (AutoProcessor, AutoTokenizer, AutoModel,
                          AutoModelForImageTextToText, MllamaForConditionalGeneration,
                          BitsAndBytesConfig)
from qwen_vl_utils import process_vision_info

dev = "cuda"
print(torch.cuda.get_device_name(0))

Tesla T4


## Llama 3.2 erişimi (gated)


In [13]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

## Modellerin teknik özellikleri

| Model | Omurga | ~Parametre | Grounding çıktı formatı |
|---|---|---|---|
| **Qwen2.5-VL-3B** | ViT + Qwen2.5 LLM | ~3.75 B | JSON `bbox_2d`, mutlak piksel |
| **InternVL2.5-4B** | InternViT + InternLM2.5 | ~4 B | `<ref>sınıf</ref><box>[[...]]</box>`, [0,1000] normalize |
| **Llama-3.2-11B-Vision** | ViT + Llama 3.1 8B + cross-attn | ~11 B | standart değil (JSON prompt'la deneniyor) |

**Not:** Bu görevde büyük = iyi değil; belirleyici olan grounding eğitimi ve koordinat formatı.
Grounding'e iyi eğitilmiş küçük bir model (Qwen-3B), grounding'i ikincil olan büyük bir modeli (Llama-11B) geçebilir.

## Veri seti - Roboflow off-road detection

[off-road detection](https://universe.roboflow.com/offroad-detection/off-road-detection) - 6 sınıf:
Trees, Bush, Rock, Car, Fence, Person. API key ücretsiz: roboflow.com > Settings > API Key.

In [6]:
!pip install -q roboflow
from roboflow import Roboflow
from google.colab import userdata


rf = Roboflow(api_key=userdata.get("Roboflow"))
ds = rf.workspace("offroad-detection").project("off-road-detection").version(1).download("coco")
DATA_DIR = ds.location + "/valid"   # valid split (29 görsel); istersen /test

loading Roboflow workspace...
loading Roboflow project...


In [7]:
def load_coco(json_path, img_root, n):
    from pycocotools.coco import COCO
    c = COCO(json_path)
    cats = {k["id"]: k["name"].lower() for k in c.loadCats(c.getCatIds())}
    images, gts = {}, {}
    for k, i in enumerate(sorted(c.getImgIds())[:n]):
        info = c.loadImgs(i)[0]
        images[k] = Image.open(os.path.join(img_root, info["file_name"])).convert("RGB")
        boxes = []
        for a in c.loadAnns(c.getAnnIds(imgIds=i)):
            x, y, w, h = a["bbox"]
            boxes.append(([x, y, x + w, y + h], cats[a["category_id"]]))
        gts[k] = boxes
    classes = sorted({lab for bs in gts.values() for _, lab in bs})
    return images, gts, classes

def load_yolo(root, n):
    import yaml
    yml = glob.glob(f"{root}/**/data.yaml", recursive=True)[0]
    names = yaml.safe_load(open(yml))["names"]
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]
    classes = [x.lower() for x in names]
    paths = sorted(glob.glob(f"{root}/**/images/*.*", recursive=True))[:n]
    images, gts = {}, {}
    for k, p in enumerate(paths):
        im = Image.open(p).convert("RGB"); W, H = im.size
        images[k] = im
        lp = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        boxes = []
        if os.path.exists(lp):
            for line in open(lp):
                v = line.split()
                if len(v) < 5:
                    continue
                cid = int(v[0]); cx, cy, bw, bh = map(float, v[1:5])
                boxes.append(([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H], classes[cid]))
        gts[k] = boxes
    return images, gts, classes

def load_data(root, n):
    js = [p for p in glob.glob(f"{root}/**/*.json", recursive=True)
          if any(t in p.lower() for t in ("annot", "instances", "coco"))]
    if js:
        return load_coco(js[0], os.path.dirname(js[0]), n)
    return load_yolo(root, n)

In [8]:
N_IMAGES = 29
images, gts, classes = load_data(DATA_DIR, N_IMAGES)
imgs = [images[k] for k in sorted(images)]
print(len(imgs), "görsel |", len(classes), "sınıf:", classes)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
29 görsel | 4 sınıf: ['bush', 'person', 'tree', 'valla']


## Modeller ve ortak ayarlar

In [9]:
MODELS = [
    {"id": "Qwen/Qwen2.5-VL-3B-Instruct",              "type": "qwen"},
    {"id": "OpenGVLab/InternVL2_5-4B",                 "type": "internvl"},
    {"id": "meta-llama/Llama-3.2-11B-Vision-Instruct", "type": "llama"},
]

WARMUP     = 1
MAX_TOKENS = 768
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

QUERIES = list(classes)   # sorgu listesi (parametre testlerinde değişir)

def class_str():
    return ", ".join(QUERIES)

## Model yükleme, InternVL ön-işleme ve predict fonksiyonları

Her VLM'in kendi arayüzü ve koordinat formatı var. Hepsi `predict(b, image)` ile çağrılıp
`(box[x1,y1,x2,y2], label, score)` listesi döndürüyor; kutular orijinal görsel piksel uzayında.

In [22]:
def to_base(lab):
    lab = lab.strip().lower()
    for c in classes:
        cc, ll = c.rstrip("s"), lab.rstrip("s")
        if cc == ll or cc in ll or ll in cc:
            return c
    return lab

# ---- InternVL görüntü ön-işleme (resmi kod) ----
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def _iv_transform(sz):
    return T.Compose([
        T.Lambda(lambda im: im.convert("RGB")),
        T.Resize((sz, sz), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def _iv_preprocess(image, sz=448, max_num=12):
    w, h = image.size; ar = w / h
    ratios = sorted({(i, j) for n in range(1, max_num+1) for i in range(1, n+1)
                     for j in range(1, n+1) if 1 <= i*j <= max_num}, key=lambda x: x[0]*x[1])
    best, bd = (1, 1), 1e9
    for r in ratios:
        d = abs(ar - r[0]/r[1])
        if d < bd:
            bd, best = d, r
    tw, th = sz*best[0], sz*best[1]
    img = image.resize((tw, th))
    cols = tw // sz
    tiles = [img.crop(((i % cols)*sz, (i//cols)*sz, (i % cols+1)*sz, (i//cols+1)*sz))
             for i in range(best[0]*best[1])]
    if best[0]*best[1] != 1:
        tiles.append(image.resize((sz, sz)))
    return tiles

def load_image_internvl(image, sz=448, max_num=12):
    tf = _iv_transform(sz)
    return torch.stack([tf(t) for t in _iv_preprocess(image, sz, max_num)])

# ---- yükleme ----
def load_model(m):
    t = m["type"]
    if t == "qwen":
        proc = AutoProcessor.from_pretrained(m["id"], min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
        model = AutoModelForImageTextToText.from_pretrained(
            m["id"], quantization_config=bnb, device_map="auto").eval()
        return {"proc": proc, "model": model, "type": t, "id": m["id"]}
    if t == "llama":
        proc = AutoProcessor.from_pretrained(m["id"])
        model = MllamaForConditionalGeneration.from_pretrained(
            m["id"], quantization_config=bnb, device_map="auto").eval()
        return {"proc": proc, "model": model, "type": t, "id": m["id"]}
    if t == "internvl":
        torch.cuda.empty_cache()
        tok = AutoTokenizer.from_pretrained(m["id"], trust_remote_code=True, use_fast=False, revision="main", force_download=True)
        try:
            # First try without quantization
            print("Attempting to load InternVL model without quantization...")
            model = AutoModel.from_pretrained(
                m["id"], trust_remote_code=True, revision="main", force_download=True).eval()
        except AttributeError as e:
            if "all_tied_weights_keys" in str(e):
                print(f"Warning: InternVL model loading without quantization failed due to '{e}'. Retrying with quantization.")
                torch.cuda.empty_cache()
                model = AutoModel.from_pretrained(
                    m["id"], quantization_config=bnb, trust_remote_code=True, revision="main", force_download=True).eval()
            else:
                raise e # Re-raise other AttributeErrors
        return {"tok": tok, "model": model, "type": t, "id": m["id"]}

def free(b):
    for k in ("model", "proc", "tok"):
        if k in b:
            del b[k]
    gc.collect(); torch.cuda.empty_cache()

In [20]:
# ---- ortak JSON parser (Qwen / Llama) ----
def parse_json_boxes(text):
    t = re.sub(r"^```(?:json)?", "", text.strip()).strip().strip("`").strip()
    data = None
    try:
        data = json.loads(t)
    except Exception:
        mm = re.search(r"\[.*\]", t, re.S)
        if mm:
            try:
                data = json.loads(mm.group())
            except Exception:
                data = None
    out = []
    if isinstance(data, list):
        for d in data:
            if not isinstance(d, dict):
                continue
            bb = d.get("bbox_2d") or d.get("bbox") or d.get("box_2d") or d.get("box")
            lab = str(d.get("label") or d.get("category") or "").lower()
            if not bb:
                for k, v in d.items():
                    if isinstance(v, (list, tuple)) and len(v) == 4:
                        bb, lab = v, str(k).lower(); break
            if bb and len(bb) == 4:
                out.append(([float(x) for x in bb], to_base(lab), 1.0))
    return out

@torch.no_grad()
def _qwen(b, image):
    prompt = ("Detect all instances of these categories in the image: " + class_str() + ". "
              'Output a JSON array; each item must be {"bbox_2d": [x1, y1, x2, y2], '
              '"label": "<category>"} using absolute pixel coordinates.')
    msgs = [{"role": "user", "content": [{"type": "image", "image": image},
                                         {"type": "text", "text": prompt}]}]
    text = b["proc"].apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    img_in, _ = process_vision_info(msgs)
    inp = b["proc"](text=[text], images=img_in, return_tensors="pt").to(b["model"].device)
    out = b["model"].generate(**inp, max_new_tokens=MAX_TOKENS, do_sample=False)
    txt = b["proc"].batch_decode(out[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return parse_json_boxes(txt)

@torch.no_grad()
def _llama(b, image):
    prompt = ("Detect all instances of these categories in the image: " + class_str() + ". "
              'Output ONLY a JSON array; each item {"bbox_2d": [x1, y1, x2, y2], "label": "<category>"} '
              "with absolute pixel coordinates (image is %dx%d)." % (image.size[0], image.size[1]))
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    txt_in = b["proc"].apply_chat_template(msgs, add_generation_prompt=True)
    inp = b["proc"](image, txt_in, return_tensors="pt").to(b["model"].device)
    out = b["model"].generate(**inp, max_new_tokens=MAX_TOKENS, do_sample=False)
    txt = b["proc"].decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_json_boxes(txt)

def parse_internvl(resp, size):
    W, H = size; out = []
    quad = r"(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)"
    for mref in re.finditer(r"<ref>(.*?)</ref>\s*<box>(.*?)</box>", resp, re.S):
        lab = to_base(mref.group(1).strip().lower())
        for q in re.findall(quad, mref.group(2)):
            x1, y1, x2, y2 = [float(v) for v in q]
            out.append(([x1/1000*W, y1/1000*H, x2/1000*W, y2/1000*H], lab, 1.0))
    if not out:                       # <ref> yoksa: "car[[...]]" biçimi
        for mm in re.finditer(r"([A-Za-z ]{2,}?)\s*(\[\[.*?\]\])", resp, re.S):
            lab = to_base(mm.group(1).strip().lower())
            for q in re.findall(quad, mm.group(2)):
                x1, y1, x2, y2 = [float(v) for v in q]
                out.append(([x1/1000*W, y1/1000*H, x2/1000*W, y2/1000*H], lab, 1.0))
    return out

@torch.no_grad()
def _internvl(b, image):
    pv = load_image_internvl(image).to(torch.bfloat16).cuda()
    q = ("<image>\nPlease detect and provide all the bounding boxes of "
         + ", ".join("<ref>%s</ref>" % c for c in QUERIES) + " in the image.")
    resp = b["model"].chat(b["tok"], pv, q, dict(max_new_tokens=MAX_TOKENS, do_sample=False))
    return parse_internvl(resp, image.size)

def predict(b, image):
    return {"qwen": _qwen, "internvl": _internvl, "llama": _llama}[b["type"]](b, image)

def warmup(b):
    for im in imgs[:WARMUP]:
        predict(b, im)
    torch.cuda.synchronize()

## Ham çıktı kontrolü (debug)

Her VLM'in ilk görseldeki ham çıktısını yazdırıyoruz. Kutu/JSON görüyorsan parser çalışır;
boş veya farklı formatsa çıktıyı paylaş, parser'ı ona göre düzeltiriz. (Her biri modeli bir kez yükler.)

In [2]:
for m in MODELS:
    b = load_model(m)
    print("="*70, "\n", m["id"])
    print(predict(b, imgs[0]))
    free(b)

NameError: name 'MODELS' is not defined

## Metrik ve görsel yardımcıları

In [1]:
colors = ["red", "cyan", "orange", "lime"]

def metric_table(d, name):
    return pd.DataFrame({name: {k.split("/")[-1]: round(v, 3) for k, v in d.items()}})

def show_preds():
    n = len(MODELS) + 1
    fig, ax = plt.subplots(len(SAMPLE), n, figsize=(3.6*n, 3.6*len(SAMPLE)))
    if len(SAMPLE) == 1:
        ax = np.array([ax])
    for r, i in enumerate(SAMPLE):
        g = imgs[i].copy(); d = ImageDraw.Draw(g)
        for box, lab in gts[i]:
            d.rectangle(box, outline="white", width=3)
        ax[r, 0].imshow(g); ax[r, 0].set_title("ground truth"); ax[r, 0].axis("off")
        for c, m in enumerate(MODELS):
            im = imgs[i].copy(); d = ImageDraw.Draw(im)
            for box, lab, sc in sample_preds[m["id"]][r]:
                d.rectangle(box, outline=colors[c], width=3)
                d.text((box[0], box[1]-10), lab, fill=colors[c])
            ax[r, c+1].imshow(im); ax[r, c+1].set_title(m["id"].split("/")[-1], fontsize=9)
            ax[r, c+1].axis("off")
    plt.tight_layout(); plt.show()

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0

def ap_per_class(preds, thr=0.5, n=None):
    n = n or len(imgs); out = {}
    for c in classes:
        entries, npos = [], 0
        for i in range(n):
            gt = [g for g in gts[i] if g[1] == c]; used = [False]*len(gt); npos += len(gt)
            for box, lab, sc in sorted([p for p in preds[i] if p[1] == c], key=lambda x: -x[2]):
                best, bi = -1, thr
                for j, (gb, _) in enumerate(gt):
                    if not used[j] and iou(box, gb) >= bi:
                        bi, best = iou(box, gb), j
                if best >= 0:
                    used[best] = True; entries.append((sc, 1))
                else:
                    entries.append((sc, 0))
        if npos == 0:
            continue
        entries.sort(key=lambda x: -x[0])
        tp = fp = 0; prec = []; rec = []
        for sc, t in entries:
            tp, fp = tp + t, fp + (1 - t)
            prec.append(tp/(tp+fp)); rec.append(tp/npos)
        out[c] = sum(max([p for p, r in zip(prec, rec) if r >= t] or [0])
                     for t in np.linspace(0, 1, 11)) / 11
    return out

def mean_ap(preds, thr=0.5, n=None):
    d = ap_per_class(preds, thr, n)
    return float(np.mean(list(d.values()))) if d else 0.0

def pr_metrics(preds, iou_thr=0.5, n=None):
    n = n or len(imgs); tp = fp = fn = 0; ious = []
    for i in range(n):
        gt = gts[i]; used = [False]*len(gt)
        for box, lab, sc in sorted(preds[i], key=lambda x: -x[2]):
            best, bi = -1, iou_thr
            for j, (gb, gc) in enumerate(gt):
                if used[j] or gc != lab:
                    continue
                v = iou(box, gb)
                if v >= bi:
                    bi, best = v, j
            if best >= 0:
                tp += 1; used[best] = True; ious.append(bi)
            else:
                fp += 1
        fn += used.count(False)
    prec = tp/(tp+fp) if tp+fp else 0.0
    recl = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*prec*recl/(prec+recl) if prec+recl else 0.0
    return {"precision": prec, "recall": recl, "f1": f1,
            "mean_iou": float(np.mean(ious)) if ious else 0.0, "tp": tp, "fp": fp, "fn": fn}

## Tek geçişli ölçüm (hepsi bir arada)

**Ağır kısım - bir kez çalışır.** Her model bir kez yükleniyor; aynı geçişte latency, FPS, tepe VRAM,
parametre sayısı ve tüm tahminler toplanıyor. Aşağıdaki hücreler sadece bu sonuçları gösterir.

In [ ]:
results, preds_all = {}, {}
for m in MODELS:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    b = load_model(m)
    params = sum(p.numel() for p in b["model"].parameters()) / 1e6
    warmup(b)
    torch.cuda.synchronize(); t0 = time.time()
    pl = [predict(b, im) for im in imgs]
    torch.cuda.synchronize(); dt = time.time() - t0
    vram = torch.cuda.max_memory_allocated() / 1e9
    free(b)
    preds_all[m["id"]] = pl
    results[m["id"]] = {"latency": dt/len(imgs)*1000, "fps": len(imgs)/dt,
                        "vram": vram, "params": params}
    print(f"{m['id']:42s} {results[m['id']]['latency']:7.1f} ms | "
          f"{results[m['id']]['fps']:5.2f} FPS | {vram:4.2f} GB | {params:7.1f} M")

latency = {k: v["latency"] for k, v in results.items()}
fps     = {k: v["fps"]     for k, v in results.items()}
vram    = {k: v["vram"]    for k, v in results.items()}
size    = {k: v["params"]  for k, v in results.items()}
mAP     = {k: mean_ap(preds_all[k]) for k in preds_all}

SAMPLE = list(range(min(3, len(imgs))))
sample_preds = {k: [preds_all[k][i] for i in SAMPLE] for k in preds_all}

## Ölçüm 1 - Latency (ms / görsel)

In [ ]:
display(metric_table(latency, "latency_ms"))
show_preds()

## Ölçüm 2 - FPS (görsel / saniye)

In [ ]:
display(metric_table(fps, "fps"))
show_preds()

## Ölçüm 3 - GPU bellek (tepe VRAM, GB)

In [ ]:
display(metric_table(vram, "vram_gb"))
show_preds()

## Ölçüm 4 - Model boyutu (milyon parametre)

> 4-bit yüklendiği için parametre sayısı yaklaşık çıkar.

In [ ]:
display(metric_table(size, "params_M"))
show_preds()

## Ölçüm 5 - mAP@0.5 (doğruluk)

> VLM'ler confidence üretmediği için skor=1.0; mAP tek çalışma noktasında hesaplanır (P/R ile birlikte yorumla).

In [ ]:
display(metric_table(mAP, "mAP@0.5"))
show_preds()

## Ölçüm 5b - Precision / Recall / IoU / sınıf bazında AP

In [ ]:
prm = {mid: pr_metrics(preds_all[mid]) for mid in preds_all}
detail = pd.DataFrame({mid.split("/")[-1]: prm[mid] for mid in prm}).T
detail = detail[["precision", "recall", "f1", "mean_iou", "tp", "fp", "fn"]].round(3)
print("Precision / Recall / F1 / ortalama IoU  (IoU eşiği 0.5)")
display(detail)

apc = pd.DataFrame({mid.split("/")[-1]: ap_per_class(preds_all[mid]) for mid in preds_all}).round(3)
print("Sınıf bazında AP@0.5")
display(apc)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
detail[["precision", "recall", "f1"]].plot(kind="bar", ax=ax[0]); ax[0].set_title("Precision / Recall / F1")
ax[0].set_xticklabels(detail.index, rotation=20, ha="right"); ax[0].grid(axis="y", alpha=0.3)
apc.T.plot(kind="bar", ax=ax[1]); ax[1].set_title("Sınıf bazında AP@0.5"); ax[1].set_ylabel("AP")
ax[1].set_xticklabels(apc.columns, rotation=20, ha="right"); ax[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## Parametre testi 1 - Input çözünürlüğü

Sadece giriş görselinin boyutunu (uzun kenar) değiştirip mAP ve latency'ye bakıyoruz.
Görsel yeniden boyutlandırılıp modele veriliyor, kutular orijinale geri ölçekleniyor.
> ⚠️ Yeniden inference yapar (Llama dahil, en yavaş kısım). Yavaşsa `SWEEP_N` / `RESOLUTIONS`'ı küçült.

In [ ]:
RESOLUTIONS = [448, 640, 896]
SWEEP_N = min(len(imgs), 12)

def detect_res(b, image, longest):
    w, h = image.size
    s = longest / max(w, h)
    rim = image.resize((max(1, int(w*s)), max(1, int(h*s))))
    return [([x1/s, y1/s, x2/s, y2/s], lab, sc)
            for (x1, y1, x2, y2), lab, sc in predict(b, rim)]

res_map, res_lat = {}, {}
for m in MODELS:
    b = load_model(m)
    maps, lats = [], []
    for R in RESOLUTIONS:
        detect_res(b, imgs[0], R)
        torch.cuda.synchronize(); t = time.time()
        preds = [detect_res(b, imgs[i], R) for i in range(SWEEP_N)]
        torch.cuda.synchronize()
        lats.append((time.time() - t) / SWEEP_N * 1000)
        maps.append(mean_ap(preds, n=SWEEP_N))
    res_map[m["id"].split("/")[-1]] = maps
    res_lat[m["id"].split("/")[-1]] = lats
    free(b)

res_map_df = pd.DataFrame(res_map, index=RESOLUTIONS); res_map_df.index.name = "input_size"
res_lat_df = pd.DataFrame(res_lat, index=RESOLUTIONS); res_lat_df.index.name = "input_size"
print("mAP@0.5"); display(res_map_df.round(3))
print("latency (ms)"); display(res_lat_df.round(1))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
res_map_df.plot(marker="o", ax=ax[0]); ax[0].set_title("input size vs mAP@0.5"); ax[0].set_ylabel("mAP@0.5")
res_lat_df.plot(marker="o", ax=ax[1]); ax[1].set_title("input size vs latency"); ax[1].set_ylabel("ms")
for a in ax:
    a.set_xlabel("uzun kenar (px)"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Parametre testi 2 - Sorgu sınıf sayısı

Prompt'a kaç sınıf koyduğun latency'yi etkiler. Sınıf sayısını değiştirip süre ölçüyoruz.
> ⚠️ Yeniden inference yapar. Yavaşsa `SWEEP_N`'i küçült.

In [ ]:
QCOUNTS = [k for k in [1, 2, 4, 6] if k <= len(classes)]
qc_rows = {}
for m in MODELS:
    b = load_model(m)
    lats = []
    for k in QCOUNTS:
        QUERIES[:] = classes[:k]
        predict(b, imgs[0])
        torch.cuda.synchronize(); t = time.time()
        for i in range(SWEEP_N):
            predict(b, imgs[i])
        torch.cuda.synchronize()
        lats.append((time.time() - t) / SWEEP_N * 1000)
    qc_rows[m["id"].split("/")[-1]] = lats
    free(b)
QUERIES[:] = list(classes)

qc_df = pd.DataFrame(qc_rows, index=QCOUNTS); qc_df.index.name = "n_queries"
display(qc_df.round(1))
qc_df.plot(marker="o", figsize=(8, 5)); plt.xlabel("sorgu sınıf sayısı"); plt.ylabel("latency (ms/görsel)")
plt.title("Sorgu sayısı vs latency"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Parametre testi 3 - IoU değerlendirme eşiği

Kutu eşleşmesinde IoU eşiğini sıkılaştırınca mAP nasıl düşüyor (lokalizasyon hassasiyeti).
Yeni inference yok, `preds_all` kullanılıyor.

In [ ]:
IOU_THRS = [0.3, 0.5, 0.7, 0.9]
iou_rows = {mid.split("/")[-1]: [mean_ap(preds_all[mid], thr=t) for t in IOU_THRS] for mid in preds_all}
iou_df = pd.DataFrame(iou_rows, index=IOU_THRS); iou_df.index.name = "IoU_thr"
display(iou_df.round(3))
iou_df.plot(marker="o", figsize=(8, 5)); plt.xlabel("IoU eşiği"); plt.ylabel("mAP")
plt.title("IoU eşiği vs mAP"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Sonuçlar

In [ ]:
df = pd.DataFrame({
    "latency_ms": latency, "fps": fps, "vram_gb": vram,
    "params_M": size, "mAP@0.5": mAP,
})
df.index = [i.split("/")[-1] for i in df.index]
df = df.round(3)
df

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, col, title in zip(ax, ["latency_ms", "fps", "vram_gb", "mAP@0.5"],
                         ["Latency (ms)", "FPS", "VRAM (GB)", "mAP@0.5"]):
    a.bar(df.index, df[col]); a.set_title(title); a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()